In [ ]:
# ============================================================
# Personalized Course Recommendation Engine
# Embeddings + Vector DB + RAG with Gemini 2.5 Flash
# ============================================================

import os
import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
import google.generativeai as genai
import mlflow
import mlflow.tracing
from typing import List, Dict, Tuple
import json
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================
# Set your API key securely (use environment variables in production)
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"  # Replace with your key
genai.configure(api_key=GEMINI_API_KEY)

# Embedding model (Google's text-embedding-004 via Gemini embedding)
EMBEDDING_MODEL = "models/text-embedding-004"

# LLM model
LLM_MODEL = "gemini-2.0-flash-exp"  # or gemini-2.5-flash if available

# Dataset URL
DATASET_URL = "https://raw.githubusercontent.com/Bluedata-Consulting/GAAPB01-training-code-base/refs/heads/main/Assignments/assignment2dataset.csv"
LOCAL_CSV = "assignment2dataset.csv"

# ChromaDB path
CHROMA_PATH = "./chroma_course_db"

# MLflow tracking
mlflow.set_experiment("Course_Recommendation_Engine")
# ========================================================

# -------------------- Step 1: Load Dataset --------------------
def load_courses(csv_path=None):
    if csv_path is None:
        csv_path = LOCAL_CSV
    if not os.path.exists(csv_path):
        print(f"Downloading dataset from {DATASET_URL}")
        df = pd.read_csv(DATASET_URL)
        df.to_csv(csv_path, index=False)
    else:
        df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} courses")
    return df

# -------------------- Step 2: Embedding Function --------------------
class GeminiEmbeddingFunction(embedding_functions.EmbeddingFunction):
    def __init__(self, model_name=EMBEDDING_MODEL):
        self.model_name = model_name

    def __call__(self, texts: List[str]) -> List[List[float]]:
        embeddings = []
        for text in texts:
            result = genai.embed_content(
                model=self.model_name,
                content=text,
                task_type="retrieval_document"
            )
            embeddings.append(result['embedding'])
        return embeddings

# -------------------- Step 3: Build Vector DB --------------------
def build_vector_db(df, collection_name="courses"):
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    # Delete existing if any
    try:
        client.delete_collection(collection_name)
    except:
        pass

    embedding_fn = GeminiEmbeddingFunction()
    collection = client.create_collection(
        name=collection_name,
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"}
    )

    # Add documents
    ids = df['course_id'].tolist()
    documents = df['description'].tolist()
    metadatas = df[['course_id', 'title']].to_dict(orient='records')

    # Batch add (Chroma handles batching)
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    print(f"Added {len(ids)} courses to vector DB")
    return collection

# -------------------- Step 4: Retrieve similar courses --------------------
def retrieve_courses(user_query, collection, completed_courses=[], top_k=5):
    # Embed query using same embedding model
    embedding_fn = GeminiEmbeddingFunction()
    query_embedding = embedding_fn([user_query])[0]

    # Search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k + len(completed_courses)  # Oversample to filter completed
    )

    # Filter out completed courses
    recommendations = []
    for i, idx in enumerate(results['ids'][0]):
        course_id = idx
        if course_id not in completed_courses:
            recommendations.append({
                "course_id": course_id,
                "title": results['metadatas'][0][i]['title'],
                "description": results['documents'][0][i],
                "similarity_score": 1 - results['distances'][0][i]  # cosine distance to similarity
            })
        if len(recommendations) == top_k:
            break

    return recommendations

# -------------------- Step 5: Generate LLM Rationale --------------------
def generate_rationale(user_query, completed_courses, recommendations, course_df):
    # Prepare context
    context = ""
    for rec in recommendations:
        context += f"- {rec['title']} (ID: {rec['course_id']}): {rec['description'][:200]}...\n"

    prompt = f"""
You are a course recommendation expert. A learner has the following background:
"{user_query}"

Courses they already completed: {', '.join(completed_courses) if completed_courses else 'None'}

Based on semantic similarity, the system retrieved these top-5 relevant courses:
{context}

For each of the 5 recommended courses, provide a short (1-2 sentence) personalized rationale explaining why this course is a good next step given their background.

Return the output as a JSON list with keys: "rank", "course_id", "title", "rationale".
"""
    model = genai.GenerativeModel(LLM_MODEL)
    response = model.generate_content(prompt)

    # Parse LLM response (robust parsing)
    try:
        # Extract JSON part
        text = response.text
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        rationales = json.loads(text)
        return rationales
    except:
        # Fallback: generate generic rationales
        fallback = []
        for i, rec in enumerate(recommendations):
            fallback.append({
                "rank": i+1,
                "course_id": rec['course_id'],
                "title": rec['title'],
                "rationale": f"This course matches your stated interests in {user_query[:50]}..."
            })
        return fallback

# -------------------- Step 6: Full Recommendation Pipeline --------------------
@mlflow.trace(name="course_recommendation_pipeline", span_type="PIPELINE")
def recommend_courses(user_query, completed_courses, collection, course_df, top_k=5):
    with mlflow.start_span(name="retrieval") as span:
        recommendations = retrieve_courses(user_query, collection, completed_courses, top_k)
        span.set_attributes({"query": user_query, "num_retrieved": len(recommendations)})

    with mlflow.start_span(name="llm_rationale") as span:
        rationales = generate_rationale(user_query, completed_courses, recommendations, course_df)
        span.set_attributes({"model": LLM_MODEL})

    # Merge scores with rationales
    output = []
    for i, rec in enumerate(recommendations):
        rationale_text = ""
        for r in rationales:
            if r.get('course_id') == rec['course_id'] or r.get('title') == rec['title']:
                rationale_text = r.get('rationale', '')
                break
        if not rationale_text and i < len(rationales):
            rationale_text = rationales[i].get('rationale', '')

        output.append({
            "rank": i+1,
            "course_id": rec['course_id'],
            "title": rec['title'],
            "similarity_score": round(rec['similarity_score'], 4),
            "rationale": rationale_text
        })

    return {
        "user_query": user_query,
        "completed_courses": completed_courses,
        "recommendations": output,
        "total_recommendations": len(output),
        "model_used": LLM_MODEL,
        "embedding_model": EMBEDDING_MODEL
    }

# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    # Load data
    df = load_courses()

    # Build or load vector DB
    collection = build_vector_db(df)

    # Test queries
    test_queries = [
        ("I've completed the 'Python Programming for Data Science' course and enjoy data visualization. What should I take next?", ["C001"]),
        ("I know Azure basics and want to manage containers and build CI/CD pipelines. Recommend courses.", []),
        ("My background is in ML fundamentals; I'd like to specialize in neural networks and production workflows.", []),
        ("I want to learn to build and deploy microservices with Kubernetes — what courses fit best?", []),
        ("I'm interested in blockchain and smart contracts but have no prior experience. Which courses do you suggest?", [])
    ]

    results = []
    for query, completed in test_queries:
        with mlflow.start_run(run_name=f"Query: {query[:30]}..."):
            result = recommend_courses(query, completed, collection, df)
            results.append(result)
            print(json.dumps(result, indent=2))
            print("\n" + "="*80 + "\n")

    # Save results
    with open("recommendation_output.json", "w") as f:
        json.dump(results, f, indent=2)

    print("Done. Results saved to recommendation_output.json")